# DataFrame Baseline — Naive Bayes Classifier

**Unoptimised implementation using the PySpark DataFrame / SQL API.**

This notebook is the baseline reference — no optimisations applied.
It uses Python UDFs wherever possible (UDFs are opaque to Spark's Catalyst
optimizer and involve Python serialisation overhead) and does not call
`.persist()` or `.repartition()`.
See `naive_bayes_df_optimized.ipynb` for the improved version.

The algorithm is identical to the RDD version: the same `compute_log_probs()`,
`predict()`, and `evaluate()` functions from `core/naive_bayes.py` are used,
ensuring that probability outputs match the RDD implementation exactly.

In [ ]:
import time
import math
import sys
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

sys.path.insert(0, os.path.abspath('..'))

from core.naive_bayes import compute_log_probs, evaluate
from data.loader import load_car_dataframe, FEATURE_COLS, LABEL_COL

# after online research, its better to implement the predict function locally
def predict(log_prob_table, test_point):
    best_class = None
    best_score = float('-inf')
    for class_label in log_prob_table['classes']:
        score = log_prob_table['log_class_probs'][class_label]
        for feat_idx, value in enumerate(test_point):
            key = f'feat_{feat_idx}_{value}_{class_label}'
            if key in log_prob_table['log_feature_probs']:
                score += log_prob_table['log_feature_probs'][key]
            else:
                score += log_prob_table['fallback_log_probs'].get(
                    (feat_idx, class_label), math.log(1e-10)
                )
        if score > best_score:
            best_score = score
            best_class = class_label
    return best_class

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('NaiveBayes-DF-Baseline')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')
print('SparkSession ready.')

SparkSession ready.


In [ ]:
# change the path to whichever file you want to load
DATA_PATH = '/Users/luccagazotto/Documents/École Polytechnique/DSAIB/Database Management/final_project/Database-Management---Naive-Bayes/data/car.csv'

# splitting the data
train_df, test_df = load_car_dataframe(spark, filepath=DATA_PATH)

train_df.groupBy(LABEL_COL).count().orderBy('count', ascending=False).show()

+-----+-----+
|label|count|
+-----+-----+
|unacc| 1008|
|  acc|  314|
| good|   58|
|vgood|   51|
+-----+-----+



26/04/02 20:07:14 WARN CacheManager: Asked to cache already cached data.


In [16]:
# training

t_train_start = time.time()

class_counts_df = train_df.groupBy(LABEL_COL).agg(F.count('*').alias('class_count'))

# baseline uses a udf (not optimized)
make_key_udf = F.udf(
    lambda feat_idx, value, label: f'feat_{feat_idx}_{value}_{label}',
    StringType()
)

# building one dataframe per feature 
key_dfs = []
for feat_idx, col_name in enumerate(FEATURE_COLS):
    key_df = train_df.select(
        make_key_udf(F.lit(feat_idx), F.col(col_name), F.col(LABEL_COL)).alias('key')
    )
    key_dfs.append(key_df)

all_keys_df = key_dfs[0]
for df in key_dfs[1:]:
    all_keys_df = all_keys_df.union(df)

# counting occurences
feature_counts_df = all_keys_df.groupBy('key').agg(F.count('*').alias('count'))

train_time = time.time() - t_train_start
print(f'Training: {train_time:.4f}s')
print(f'Unique feature keys: {feature_counts_df.count()}')

Training: 0.0914s
Unique feature keys: 69


In [17]:
# prob computation

class_counts_dict   = {row[LABEL_COL]: row['class_count'] for row in class_counts_df.collect()}
feature_counts_dict = {row['key']: row['count'] for row in feature_counts_df.collect()}
class_totals        = class_counts_dict

log_prob_table = compute_log_probs(class_counts_dict,
                                   feature_counts_dict, class_totals)

print(f"Classes           : {log_prob_table['classes']}")
print(f"Log class probs   : {log_prob_table['log_class_probs']}")
print(f"Sample feat probs : {list(log_prob_table['log_feature_probs'].items())[:3]}")

Classes           : ['unacc', 'acc', 'vgood', 'good']
Log class probs   : {'unacc': -0.35220510784011255, 'acc': -1.5163474893680884, 'vgood': -3.317676409612294, 'good': -3.191382684288002}
Sample feat probs : [('feat_0_vhigh_acc', -1.6189166563886441), ('feat_0_low_acc', -1.5279448781829175), ('feat_0_med_acc', -1.208174491179636)]


In [ ]:
# prediction

predict_udf = F.udf(
    lambda *features: predict(log_prob_table, list(features)),
    StringType()
)

t_pred_start = time.time()

prediction_df = test_df.withColumn(
    'prediction',
    predict_udf(*[F.col(c) for c in FEATURE_COLS])
)

results = prediction_df.select(LABEL_COL, 'prediction').collect()

pred_time = time.time() - t_pred_start
print(f'[TIMING] Prediction: {pred_time:.4f}s')
print(f'Sample: {results[:5]}')

[TIMING] Prediction: 0.0513s
Sample: [Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc'), Row(label='unacc', prediction='unacc')]


In [18]:
# evaluation

true_labels = [row[LABEL_COL] for row in results]
pred_labels = [row['prediction'] for row in results]

accuracy = evaluate(pred_labels, true_labels)

print(f'Accuracy     : {accuracy:.4f} ({accuracy * 100:.2f}%)')
print(f'Train time   : {train_time:.4f}s')
print(f'Predict time : {pred_time:.4f}s')
print(f'Total time   : {train_time + pred_time:.4f}s')

Accuracy     : 0.8552 (85.52%)
Train time   : 0.0914s
Predict time : 0.0513s
Total time   : 0.1427s
